# RetailPulse -- Exploratory Data Analysis

**Dataset:** Online Retail II (UCI Machine Learning Repository)

**Objective:** Distribution analysis, missing values assessment, correlation heatmap, and initial data profiling for the retail analytics platform.


In [1]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")


In [2]:
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
})

FIGURES_DIR = os.path.join("..", "reports", "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)


In [3]:
def save_fig(fig, name):
    path = os.path.join(FIGURES_DIR, name)
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"Saved: {name}")


## Data Loading

In [4]:
DATA_PATH = os.path.join("..", "data", "raw", "online_retail_II.csv")

df = pd.read_csv(DATA_PATH, parse_dates=["InvoiceDate"])
print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")


Dataset shape: 525,461 rows x 8 columns


Memory usage: 146.6 MB


In [5]:
df.head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom
8,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,13085.0,United Kingdom


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 525461 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      525461 non-null  object        
 1   StockCode    525461 non-null  object        
 2   Description  522533 non-null  object        
 3   Quantity     525461 non-null  int64         
 4   InvoiceDate  525461 non-null  datetime64[ns]
 5   Price        525461 non-null  float64       
 6   Customer ID  417534 non-null  float64       
 7   Country      525461 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 32.1+ MB


## Basic Data Profiling

In [7]:
df.describe().round(2)

,Quantity,InvoiceDate,Price,Customer ID
count,525461.00,525461,525461.00,417534.00
mean,10.34,2010-06-28 11:37:36.845017856,4.69,15360.65
min,-9600.00,2009-12-01 07:45:00,-53594.36,12346.00
25%,1.00,2010-03-21 12:20:00,1.25,13983.00
50%,3.00,2010-07-06 09:51:00,2.10,15311.00
75%,10.00,2010-10-15 12:45:00,4.21,16799.00
max,19152.00,2010-12-09 20:01:00,25111.09,18287.00
std,107.42,NaN,146.13,1680.81


In [8]:
df.describe(include="object")

,Invoice,StockCode,Description,Country
count,525461,525461,522533,525461
unique,28816,4632,4681,40
top,537434,85123A,WHITE HANGING HEART T-LIGHT HOLDER,United Kingdom
freq,675,3516,3549,485852


In [9]:
print(f"Date range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")
print(f"Unique invoices: {df['Invoice'].nunique():,}")
print(f"Unique products (StockCode): {df['StockCode'].nunique():,}")
print(f"Unique customers: {df['Customer ID'].nunique():,}")
print(f"Countries: {df['Country'].nunique()}")


Date range: 2009-12-01 07:45:00 to 2010-12-09 20:01:00
Unique invoices: 28,816
Unique products (StockCode): 4,632
Unique customers: 4,383
Countries: 40


## Missing Values Analysis

In [10]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    "Missing Count": missing,
    "Missing %": missing_pct
}).sort_values("Missing %", ascending=False)

missing_df[missing_df["Missing Count"] > 0]


,Missing Count,Missing %
Customer ID,107927,20.54
Description,2928,0.56


In [11]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

cols_with_missing = missing_df[missing_df["Missing Count"] > 0]

if len(cols_with_missing) > 0:
    colors = ["#e74c3c" if pct > 20 else "#f39c12" if pct > 5 else "#2ecc71"
              for pct in cols_with_missing["Missing %"]]
    axes[0].barh(cols_with_missing.index, cols_with_missing["Missing %"], color=colors)
    axes[0].set_xlabel("Missing (%)")
    axes[0].set_title("Missing Values by Column")
    for i, (idx, row) in enumerate(cols_with_missing.iterrows()):
        axes[0].text(row["Missing %"] + 0.3, i,
                     f'{row["Missing %"]:.1f}% ({int(row["Missing Count"]):,})',
                     va="center", fontsize=10)

# Missing pattern matrix (sample)
sample = df.sample(min(1000, len(df)), random_state=42).isnull().astype(int)
sns.heatmap(sample[cols_with_missing.index] if len(cols_with_missing) > 0 else sample,
            cbar=True, yticklabels=False, cmap="YlOrRd", ax=axes[1])
axes[1].set_title("Missing Value Pattern (Sample of 1000 rows)")

fig.suptitle("Missing Values Analysis", fontsize=16, fontweight="bold", y=1.02)
fig.tight_layout()
save_fig(fig, "01_missing_values_analysis.png")
plt.show()


Saved: 01_missing_values_analysis.png


## Distribution Analysis

In [12]:
df["Revenue"] = df["Quantity"] * df["Price"]


In [13]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Quantity distribution (filtered to 1-50 for visibility)
qty_filtered = df[(df["Quantity"] > 0) & (df["Quantity"] <= 50)]
axes[0].hist(qty_filtered["Quantity"], bins=50, color="#3498db", edgecolor="white", alpha=0.8)
axes[0].set_title("Quantity Distribution (1-50)")
axes[0].set_xlabel("Quantity")
axes[0].set_ylabel("Frequency")
axes[0].axvline(qty_filtered["Quantity"].median(), color="#e74c3c",
                linestyle="--", label=f'Median: {qty_filtered["Quantity"].median():.0f}')
axes[0].legend()

# Price distribution
price_filtered = df[(df["Price"] > 0) & (df["Price"] <= 20)]
axes[1].hist(price_filtered["Price"], bins=50, color="#2ecc71", edgecolor="white", alpha=0.8)
axes[1].set_title("Price Distribution (0-20)")
axes[1].set_xlabel("Price")
axes[1].set_ylabel("Frequency")
axes[1].axvline(price_filtered["Price"].median(), color="#e74c3c",
                linestyle="--", label=f'Median: {price_filtered["Price"].median():.2f}')
axes[1].legend()

# Revenue distribution
rev_filtered = df[(df["Revenue"] > 0) & (df["Revenue"] <= 500)]
axes[2].hist(rev_filtered["Revenue"], bins=50, color="#9b59b6", edgecolor="white", alpha=0.8)
axes[2].set_title("Revenue Distribution (0-500)")
axes[2].set_xlabel("Revenue")
axes[2].set_ylabel("Frequency")
axes[2].axvline(rev_filtered["Revenue"].median(), color="#e74c3c",
                linestyle="--", label=f'Median: {rev_filtered["Revenue"].median():.2f}')
axes[2].legend()

fig.suptitle("Distribution Analysis -- Key Numeric Variables", fontsize=16, fontweight="bold", y=1.02)
fig.tight_layout()
save_fig(fig, "02_distribution_analysis.png")
plt.show()


Saved: 02_distribution_analysis.png


In [14]:
for col in ["Quantity", "Price", "Revenue"]:
    data = df[col]
    skew = data.skew()
    kurt = data.kurtosis()
    print(f"{col:12s}  |  mean: {data.mean():>10.2f}  |  median: {data.median():>10.2f}  "
          f"|  skew: {skew:>8.2f}  |  kurtosis: {kurt:>10.2f}")


Quantity      |  mean:      10.34  |  median:       3.00  |  skew:    36.04  |  kurtosis:    6277.67
Price         |  mean:       4.69  |  median:       2.10  |  skew:  -140.77  |  kurtosis:   64868.34
Revenue       |  mean:      18.15  |  median:       9.95  |  skew:  -140.20  |  kurtosis:   45029.95


## Outlier Detection

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
numeric_vars = ["Quantity", "Price", "Revenue"]

for i, col in enumerate(numeric_vars):
    data = df[col]
    q1, q99 = data.quantile(0.01), data.quantile(0.99)
    clipped = data[(data >= q1) & (data <= q99)]

    bp = axes[i].boxplot(clipped, vert=True, patch_artist=True,
                         boxprops=dict(facecolor="#3498db", alpha=0.6),
                         medianprops=dict(color="#e74c3c", linewidth=2))
    axes[i].set_title(f"{col} (1st-99th percentile)")
    axes[i].set_ylabel(col)

    outlier_count = len(data) - len(clipped)
    axes[i].text(0.95, 0.95, f"Outliers excluded: {outlier_count:,}",
                 transform=axes[i].transAxes, ha="right", va="top",
                 fontsize=9, style="italic", color="#666")

fig.suptitle("Box Plots -- Outlier Detection (1st-99th Percentile)", fontsize=16, fontweight="bold", y=1.02)
fig.tight_layout()
save_fig(fig, "03_boxplot_outliers.png")
plt.show()


Saved: 03_boxplot_outliers.png


## Time Series Patterns

In [16]:
df["Date"] = df["InvoiceDate"].dt.date
df["Date"] = pd.to_datetime(df["Date"])
df["DayOfWeek"] = df["InvoiceDate"].dt.day_name()
df["Hour"] = df["InvoiceDate"].dt.hour
df["Month"] = df["InvoiceDate"].dt.to_period("M")

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# Daily revenue
daily_revenue = df[df["Revenue"] > 0].groupby("Date")["Revenue"].sum().reset_index()
axes[0, 0].plot(daily_revenue["Date"], daily_revenue["Revenue"], color="#3498db", alpha=0.7, linewidth=0.8)
axes[0, 0].set_title("Daily Revenue Trend")
axes[0, 0].set_xlabel("Date")
axes[0, 0].set_ylabel("Revenue")
rolling_avg = daily_revenue["Revenue"].rolling(30).mean()
axes[0, 0].plot(daily_revenue["Date"], rolling_avg, color="#e74c3c", linewidth=2, label="30-day MA")
axes[0, 0].legend()

# Monthly revenue
monthly = df[df["Revenue"] > 0].groupby("Month")["Revenue"].sum()
monthly.index = monthly.index.to_timestamp()
axes[0, 1].bar(range(len(monthly)), monthly.values, color="#2ecc71", alpha=0.8)
axes[0, 1].set_xticks(range(len(monthly)))
axes[0, 1].set_xticklabels([d.strftime("%b %Y") for d in monthly.index], rotation=45, ha="right")
axes[0, 1].set_title("Monthly Revenue")
axes[0, 1].set_ylabel("Revenue")

# Day of week pattern
dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
dow = df[df["Revenue"] > 0].groupby("DayOfWeek")["Revenue"].mean().reindex(dow_order)
colors = ["#e74c3c" if d in ["Saturday", "Sunday"] else "#3498db" for d in dow_order]
axes[1, 0].bar(range(len(dow)), dow.values, color=colors, alpha=0.8)
axes[1, 0].set_xticks(range(len(dow)))
axes[1, 0].set_xticklabels([d[:3] for d in dow_order])
axes[1, 0].set_title("Average Revenue by Day of Week")
axes[1, 0].set_ylabel("Avg Revenue per Transaction")

# Hourly pattern
hourly = df[df["Revenue"] > 0].groupby("Hour")["Revenue"].mean()
axes[1, 1].bar(hourly.index, hourly.values, color="#9b59b6", alpha=0.8)
axes[1, 1].set_title("Average Revenue by Hour of Day")
axes[1, 1].set_xlabel("Hour")
axes[1, 1].set_ylabel("Avg Revenue per Transaction")

fig.suptitle("Time Series Patterns", fontsize=16, fontweight="bold", y=1.02)
fig.tight_layout()
save_fig(fig, "04_timeseries_patterns.png")
plt.show()


Saved: 04_timeseries_patterns.png


In [17]:
print(f"Total days in dataset: {daily_revenue['Date'].nunique()}")
print(f"Avg daily revenue: {daily_revenue['Revenue'].mean():,.2f}")
print(f"Peak day: {daily_revenue.loc[daily_revenue['Revenue'].idxmax(), 'Date'].date()} "
      f"({daily_revenue['Revenue'].max():,.2f})")


Total days in dataset: 307
Avg daily revenue: 33,570.90
Peak day: 2010-09-27 (118,909.67)


## Correlation Analysis

In [18]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
cmap = sns.diverging_palette(250, 15, s=75, l=40, n=9, center="light", as_cmap=True)
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".3f", cmap=cmap,
            center=0, square=True, linewidths=1, ax=ax,
            cbar_kws={"shrink": 0.8, "label": "Correlation"})
ax.set_title("Correlation Heatmap -- Numeric Variables", fontsize=16, fontweight="bold", pad=20)
fig.tight_layout()
save_fig(fig, "05_correlation_heatmap.png")
plt.show()


Saved: 05_correlation_heatmap.png


In [19]:
corr_matrix.round(3)

,Quantity,Price,Customer ID,Revenue,Hour
Quantity,1.000,-0.002,-0.012,0.156,-0.016
Price,-0.002,1.000,-0.003,0.453,0.003
Customer ID,-0.012,-0.003,1.000,-0.009,0.059
Revenue,0.156,0.453,-0.009,1.000,-0.016
Hour,-0.016,0.003,0.059,-0.016,1.000


## Top Products & Countries

In [20]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Top 15 products by revenue
top_products = (df[df["Revenue"] > 0]
                .groupby("Description")["Revenue"]
                .sum()
                .nlargest(15)
                .sort_values())
axes[0].barh(range(len(top_products)), top_products.values, color="#3498db", alpha=0.8)
axes[0].set_yticks(range(len(top_products)))
axes[0].set_yticklabels([desc[:30] for desc in top_products.index], fontsize=9)
axes[0].set_xlabel("Total Revenue")
axes[0].set_title("Top 15 Products by Revenue")

# Top 10 countries by revenue
top_countries = (df[df["Revenue"] > 0]
                 .groupby("Country")["Revenue"]
                 .sum()
                 .nlargest(10)
                 .sort_values())
colors = ["#e74c3c" if c == "United Kingdom" else "#3498db" for c in top_countries.index]
axes[1].barh(range(len(top_countries)), top_countries.values, color=colors, alpha=0.8)
axes[1].set_yticks(range(len(top_countries)))
axes[1].set_yticklabels(top_countries.index, fontsize=10)
axes[1].set_xlabel("Total Revenue")
axes[1].set_title("Top 10 Countries by Revenue")

fig.suptitle("Revenue Breakdown -- Products & Geography", fontsize=16, fontweight="bold", y=1.02)
fig.tight_layout()
save_fig(fig, "06_top_products_countries.png")
plt.show()


Saved: 06_top_products_countries.png


## Customer Analysis Preview

In [21]:
customers_with_id = df.dropna(subset=["Customer ID"])
print(f"Transactions with Customer ID: {len(customers_with_id):,} "
      f"({len(customers_with_id)/len(df)*100:.1f}%)")
print(f"Transactions without Customer ID: {df['Customer ID'].isna().sum():,} "
      f"({df['Customer ID'].isna().sum()/len(df)*100:.1f}%)")

customer_stats = (customers_with_id[customers_with_id["Revenue"] > 0]
                  .groupby("Customer ID")
                  .agg(
                      total_orders=("Invoice", "nunique"),
                      total_revenue=("Revenue", "sum"),
                      avg_order_value=("Revenue", "mean"),
                      total_items=("Quantity", "sum"),
                  ))

print(f"\nTotal unique customers: {len(customer_stats):,}")
print(f"Avg orders per customer: {customer_stats['total_orders'].mean():.1f}")
print(f"Avg revenue per customer: {customer_stats['total_revenue'].mean():,.2f}")
print(f"Median revenue per customer: {customer_stats['total_revenue'].median():,.2f}")


Transactions with Customer ID: 417,534 (79.5%)
Transactions without Customer ID: 107,927 (20.5%)



Total unique customers: 4,312
Avg orders per customer: 4.5
Avg revenue per customer: 2,048.24
Median revenue per customer: 706.02


In [22]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Customer revenue distribution
rev_clipped = customer_stats["total_revenue"].clip(
    upper=customer_stats["total_revenue"].quantile(0.95))
axes[0].hist(rev_clipped, bins=50, color="#3498db", edgecolor="white", alpha=0.8)
axes[0].set_title("Customer Revenue Distribution (up to 95th percentile)")
axes[0].set_xlabel("Total Revenue")
axes[0].set_ylabel("Number of Customers")

# Customer order frequency
orders_clipped = customer_stats["total_orders"].clip(upper=30)
axes[1].hist(orders_clipped, bins=30, color="#2ecc71", edgecolor="white", alpha=0.8)
axes[1].set_title("Customer Order Frequency Distribution")
axes[1].set_xlabel("Number of Orders")
axes[1].set_ylabel("Number of Customers")

fig.suptitle("Customer Behavior Overview", fontsize=16, fontweight="bold", y=1.02)
fig.tight_layout()
save_fig(fig, "07_customer_analysis_preview.png")
plt.show()


Saved: 07_customer_analysis_preview.png


## Data Quality Summary

In [23]:
negative_qty = (df["Quantity"] < 0).sum()
zero_price = (df["Price"] == 0).sum()
cancelled = df["Invoice"].astype(str).str.startswith("C").sum()

print("DATASET OVERVIEW")
print("-" * 60)
print(f"  {df.shape[0]:,} transactions across {df['Invoice'].nunique():,} invoices")
print(f"  {df['StockCode'].nunique():,} unique products, {df['Customer ID'].nunique():,} customers")
print(f"  Date range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")
print(f"  {df['Country'].nunique()} countries")
print()
print("DATA QUALITY FLAGS")
print("-" * 60)
print(f"  Missing Customer ID: {df['Customer ID'].isna().sum():,} rows ({df['Customer ID'].isna().mean()*100:.1f}%)")
print(f"  Missing Description: {df['Description'].isna().sum():,} rows ({df['Description'].isna().mean()*100:.1f}%)")
print(f"  Negative quantities (returns/cancellations): {negative_qty:,} rows")
print(f"  Zero-price items: {zero_price:,} rows")
print(f"  Cancelled invoices (prefix C): {cancelled:,} rows")
print()
print("KEY INSIGHTS")
print("-" * 60)
print("  Revenue is heavily right-skewed -- few high-value transactions drive most revenue")
print("  UK dominates sales but international markets show growth potential")
print("  Clear weekly and hourly patterns in purchasing behavior")
print("  ~25% missing Customer IDs needs handling for segmentation")
print("  Returns/cancellations need separate treatment in forecasting")


DATASET OVERVIEW
------------------------------------------------------------
  525,461 transactions across 28,816 invoices


  4,632 unique products, 4,383 customers
  Date range: 2009-12-01 07:45:00 to 2010-12-09 20:01:00
  40 countries

DATA QUALITY FLAGS
------------------------------------------------------------
  Missing Customer ID: 107,927 rows (20.5%)


  Missing Description: 2,928 rows (0.6%)
  Negative quantities (returns/cancellations): 12,326 rows
  Zero-price items: 3,687 rows
  Cancelled invoices (prefix C): 10,206 rows

KEY INSIGHTS
------------------------------------------------------------
  Revenue is heavily right-skewed -- few high-value transactions drive most revenue
  UK dominates sales but international markets show growth potential
  Clear weekly and hourly patterns in purchasing behavior
  ~25% missing Customer IDs needs handling for segmentation
  Returns/cancellations need separate treatment in forecasting


In [24]:
print("All figures saved to reports/figures/")
print()
print("Next steps:")
print("  - Data cleaning and preprocessing pipeline")
print("  - Feature engineering (RFM scores, rolling statistics)")
print("  - Data validation with quality checks")


All figures saved to reports/figures/

Next steps:
  - Data cleaning and preprocessing pipeline
  - Feature engineering (RFM scores, rolling statistics)
  - Data validation with quality checks
